In [4]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GPU DETECTION & CONFIGURATION
# ==============================================================================
import os
import gc
import re
import difflib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    PowerTransformer,
    FunctionTransformer,
    PolynomialFeatures,
)
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
import optuna
from catboost import CatBoostClassifier, Pool
from catboost.utils import get_gpu_device_count

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

# Auto-detect Kaggle or local directory path
DATA_DIR = Path(".")
if not (DATA_DIR / "test.csv").exists():
    for p in Path("/kaggle/input").rglob("test.csv"):
        DATA_DIR = p.parent
        break

# Detect GPU directly via CatBoost's native CUDA check
try:
    USE_GPU = get_gpu_device_count() > 0
except Exception:
    USE_GPU = False

print(f"Data Directory : {DATA_DIR.resolve()}")
print(f"GPU Available  : {USE_GPU}")


# ==============================================================================
# CELL 2: MEMORY OPTIMIZATION & DATA LOADING
# ==============================================================================
def reduce_mem_usage(df: pd.DataFrame, name: str = "DataFrame") -> pd.DataFrame:
    """Downcast numeric columns to float32/int32/int16 to prevent RAM bottlenecks."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type) and col_type != bool:
            c_min, c_max = df[col].min(), df[col].max()
            if pd.api.types.is_integer_dtype(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"[{name}] Memory reduced: {start_mem:.1f} MB -> {end_mem:.1f} MB")
    return df


print("\nLoading supplementary and transaction tables...")
calendar_df = pd.read_csv(DATA_DIR / "calendar.csv")
customers_df = pd.read_csv(DATA_DIR / "customers.csv")
products_df = pd.read_csv(DATA_DIR / "products.csv")
stores_df = pd.read_csv(DATA_DIR / "stores.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_path = DATA_DIR / "train.csv"
if train_path.exists():
    train_df = pd.read_csv(train_path)
    print(f"Loaded train.csv: {train_df.shape}")
else:
    raise FileNotFoundError(
        "train.csv not found in DATA_DIR. Ensure train.csv is in the working directory."
    )

print(f"Loaded test.csv : {test_df.shape}")
print(f"Target Return Rate in Train: {train_df['returned'].mean():.5f}")


# ==============================================================================
# CELL 3: CANONICAL STRING CLEANING, DEDUPLICATION & SUPPLEMENTARY TABLES
# ==============================================================================
def build_canonical_mapper(dirty_series: pd.Series, canonical_values: list) -> dict:
    """
    Deterministically map dirty/typo string labels (e.g., 'Brazil-Sutheast', 'home_ecor')
    to their exact canonical values using normalized alphanumeric keys + sequence similarity.
    """
    norm_to_canon = {
        re.sub(r"[^a-z0-9]", "", c.lower()): c for c in canonical_values
    }
    norm_keys = list(norm_to_canon.keys())
    mapping = {}
    for val in dirty_series.dropna().unique():
        clean_key = re.sub(r"[^a-z0-9]", "", str(val).lower())
        if clean_key in norm_to_canon:
            mapping[val] = norm_to_canon[clean_key]
        else:
            matches = difflib.get_close_matches(clean_key, norm_keys, n=1, cutoff=0.6)
            mapping[val] = norm_to_canon[matches[0]] if matches else str(val).strip()
    return mapping


# --- 3A. Clean & Deduplicate customers.csv ---
CANONICAL_REGIONS = sorted(stores_df["region"].dropna().unique().tolist())
region_mapper = build_canonical_mapper(customers_df["region"], CANONICAL_REGIONS)
customers_df["region"] = customers_df["region"].map(region_mapper).fillna("Unknown")

customers_df["signup_date"] = pd.to_datetime(customers_df["signup_date"], errors="coerce")
customers_df["record_updated_at"] = pd.to_datetime(customers_df["record_updated_at"], errors="coerce")

# Sort chronologically by customer_id and record_updated_at
customers_df = customers_df.sort_values(["customer_id", "record_updated_at"])

# Extract customer history features across duplicate profile records before deduplicating
cust_dup_stats = (
    customers_df.groupby("customer_id", sort=False)
    .agg(
        cust_record_count=("loyalty_tier", "size"),
        cust_loyalty_nunique=("loyalty_tier", "nunique"),
    )
    .reset_index()
)
cust_dup_stats["cust_loyalty_changed"] = (cust_dup_stats["cust_loyalty_nunique"] > 1).astype(np.int8)

# Backfill/forward-fill missing age_band across duplicate records of the same customer_id
customers_df["age_band"] = (
    customers_df.groupby("customer_id", sort=False)["age_band"]
    .transform(lambda s: s.ffill().bfill())
)

# Keep only the most recent profile update per customer_id (204,930 -> 201,901 unique customers)
customers_df = customers_df.drop_duplicates(subset=["customer_id"], keep="last").copy()
customers_df = customers_df.merge(
    cust_dup_stats[["customer_id", "cust_record_count", "cust_loyalty_changed"]],
    on="customer_id",
    how="left",
)

# Analytically impute remaining missing age_band via hierarchical mode on customer segment attributes
customers_df["age_band_missing"] = customers_df["age_band"].isna().astype(np.int8)
age_mode_table = (
    customers_df.dropna(subset=["age_band"])
    .groupby(["customer_type", "loyalty_tier", "preferred_contact"])["age_band"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={"age_band": "imputed_age_band"})
)
customers_df = customers_df.merge(
    age_mode_table, on=["customer_type", "loyalty_tier", "preferred_contact"], how="left"
)
customers_df["age_band"] = (
    customers_df["age_band"]
    .fillna(customers_df["imputed_age_band"])
    .fillna("25-34")
)
customers_df.drop(columns=["imputed_age_band"], inplace=True)

AGE_MAP = {"18-24": 21.0, "25-34": 29.5, "35-44": 39.5, "45-54": 49.5, "55-64": 59.5, "65+": 70.0}
LOYALTY_MAP = {"bronze": 1, "silver": 2, "gold": 3, "platinum": 4}
customers_df["cust_age_num"] = customers_df["age_band"].map(AGE_MAP).astype(np.float32)
customers_df["cust_loyalty_num"] = customers_df["loyalty_tier"].map(LOYALTY_MAP).fillna(1).astype(np.int8)
customers_df = customers_df.rename(columns={"region": "cust_region"})


# --- 3B. Clean & Physical-Impute products.csv ---
CANONICAL_CATEGORIES = [
    "home_decor", "kitchen_dining", "sports_leisure", "toys_gifts",
    "fashion_accessories", "office_supplies", "health_beauty", "technology",
    "electronics_accessories", "garden_diy", "furniture", "food_beverages",
]
cat_mapper = build_canonical_mapper(products_df["category"], CANONICAL_CATEGORIES)
products_df["category"] = products_df["category"].map(cat_mapper).fillna("home_decor")
products_df["launch_date"] = pd.to_datetime(products_df["launch_date"], errors="coerce")

# Compute physical volume and surface area
products_df["prod_volume_cm3"] = (
    products_df["length_cm"] * products_df["width_cm"] * products_df["height_cm"]
).clip(lower=1.0)
products_df["prod_surface_area_cm2"] = 2.0 * (
    products_df["length_cm"] * products_df["width_cm"]
    + products_df["length_cm"] * products_df["height_cm"]
    + products_df["width_cm"] * products_df["height_cm"]
)
products_df["prod_max_dim_cm"] = products_df[["length_cm", "width_cm", "height_cm"]].max(axis=1)
products_df["prod_min_dim_cm"] = products_df[["length_cm", "width_cm", "height_cm"]].min(axis=1)
products_df["prod_aspect_ratio"] = products_df["prod_max_dim_cm"] / (products_df["prod_min_dim_cm"] + 1e-3)

# Treat weight_kg <= 0 as missing and physically reconstruct from volume * median category density
products_df["weight_kg_missing"] = (
    products_df["weight_kg"].isna() | (products_df["weight_kg"] <= 0)
).astype(np.int8)
products_df.loc[products_df["weight_kg"] <= 0, "weight_kg"] = np.nan
products_df["raw_density"] = products_df["weight_kg"] / products_df["prod_volume_cm3"]

density_lookup = (
    products_df.groupby(["category", "packaging_type"])["raw_density"]
    .transform("median")
    .fillna(products_df.groupby("category")["raw_density"].transform("median"))
    .fillna(products_df["raw_density"].median())
)
products_df["weight_kg"] = products_df["weight_kg"].fillna(
    (products_df["prod_volume_cm3"] * density_lookup).clip(lower=0.05, upper=45.0)
)
products_df["prod_density_kg_cm3"] = products_df["weight_kg"] / products_df["prod_volume_cm3"]
products_df["prod_price_per_kg"] = products_df["list_price"] / (products_df["weight_kg"] + 0.01)

# Category-level relative price and weight benchmarks
products_df["cat_mean_list_price"] = products_df.groupby("category")["list_price"].transform("mean")
products_df["prod_price_vs_cat_ratio"] = products_df["list_price"] / (products_df["cat_mean_list_price"] + 1e-4)
products_df.drop(columns=["raw_density"], inplace=True)


# --- 3C. Clean & Impute stores.csv ---
stores_df["opened_date"] = pd.to_datetime(stores_df["opened_date"], errors="coerce")
stores_df["is_physical_store"] = stores_df["size_sqm"].notna().astype(np.int8)
format_median_sqm = stores_df.groupby("format")["size_sqm"].transform("median")
stores_df["size_sqm_structural"] = stores_df["size_sqm"].fillna(format_median_sqm).fillna(0.0)
stores_df["size_sqm_ratio_to_format"] = np.where(
    stores_df["size_sqm_structural"] > 0,
    stores_df["size_sqm_structural"] / (format_median_sqm.fillna(1.0) + 1e-3),
    0.0,
)
stores_df = stores_df.rename(
    columns={"channel": "store_channel", "region": "store_region"}
)
stores_df.drop(columns=["size_sqm"], inplace=True)


# --- 3D. Enrich calendar.csv with Rolling Macroeconomic Momentum ---
calendar_df["week_start"] = pd.to_datetime(calendar_df["week_start"])
calendar_df = calendar_df.sort_values("week_start").reset_index(drop=True)

for col in ["cpi", "fuel_price", "temperature_f", "unemployment", "fuel_surcharge"]:
    calendar_df[f"{col}_diff_1w"] = calendar_df[col].diff(1).fillna(0.0)
    lag4 = calendar_df[col].shift(4)
    calendar_df[f"{col}_pct_4w"] = ((calendar_df[col] - lag4) / (lag4.abs() + 1e-4)).fillna(0.0)
    roll_mean = calendar_df[col].rolling(8, min_periods=1).mean()
    roll_std = calendar_df[col].rolling(8, min_periods=1).std().fillna(1.0) + 1e-4
    calendar_df[f"{col}_zscore_8w"] = (calendar_df[col] - roll_mean) / roll_std

calendar_df["rolling_holiday_3w"] = (
    calendar_df["holiday_flag"].rolling(3, center=True, min_periods=1).mean()
)

print("Supplementary tables cleaned and enriched successfully.")


# ==============================================================================
# CELL 4: COMBINED ANALYTICAL IMPUTATION, OUTLIER CORRECTION & FEATURE ENGINEERING
# ==============================================================================
n_train = len(train_df)
y_train = train_df["returned"].values.astype(np.int8)

# Concatenate train and test for consistent relational imputation & non-target feature engineering
df = pd.concat([train_df.drop(columns=["returned"]), test_df], axis=0, ignore_index=True)
del train_df, test_df
gc.collect()

print(f"\nCombined transaction table shape: {df.shape}")

# --- 4A. Merge Supplementary Tables ---
df["order_ts"] = pd.to_datetime(df["order_ts"], errors="coerce")
df["week_start"] = df["order_ts"].dt.to_period("W-SUN").dt.start_time

df = df.merge(customers_df, on="customer_id", how="left")
df = df.merge(products_df, on="product_id", how="left")
df = df.merge(stores_df, on="store_id", how="left")
df = pd.merge_asof(
    df.sort_values("week_start"),
    calendar_df.sort_values("week_start"),
    on="week_start",
    direction="nearest",
).sort_index()

del customers_df, products_df, stores_df, calendar_df
gc.collect()

# --- 4B. Fix the 2% 'unit_price' x100 Cents-vs-Dollars Anomaly ---
raw_price_to_list = df["unit_price"] / (df["list_price"] + 1e-4)
raw_line_to_order_val = (df["unit_price"] * df["quantity"]) / (df["order_value"] + 1.0)

is_x100_glitch = (raw_price_to_list > 10.0) & (raw_line_to_order_val > 15.0)
df["is_unit_price_x100_glitch"] = is_x100_glitch.astype(np.int8)
df["unit_price"] = np.where(is_x100_glitch, df["unit_price"] / 100.0, df["unit_price"])
print(f"Corrected {is_x100_glitch.sum():,} rows with x100 unit_price cents anomaly.")

# Fix rare zero order_value rows using clean unit_price * quantity
df["order_value_was_zero"] = (df["order_value"] <= 0).astype(np.int8)
df["order_value"] = np.where(
    df["order_value"] <= 0, df["unit_price"] * df["quantity"], df["order_value"]
)

# --- 4C. Analytical Imputation of Missing 'discount' (r = 0.9735 with 1 - unit_price/list_price) ---
df["discount_missing"] = df["discount"].isna().astype(np.int8)
implied_discount = (1.0 - (df["unit_price"] / (df["list_price"] + 1e-4))).clip(0.0, 0.80)
df["discount"] = df["discount"].fillna(implied_discount)
df["discount_discrepancy"] = df["discount"] - implied_discount

# --- 4D. Order-Level & Relational Imputation of Transaction Nulls (Pandas 3.0 Safe) ---
for col in ["payment_method", "ship_mode", "tz_hint", "session_clicks", "coupon_code"]:
    df[f"{col}_missing"] = df[col].isna().astype(np.int8)

# 1. Propagate order-level attributes across lines belonging to the same order_id
for col in ["payment_method", "ship_mode", "tz_hint", "session_clicks"]:
    order_first = df.groupby("order_id", sort=False)[col].transform("first")
    df[col] = df[col].fillna(order_first)

# 2. Relationally impute remaining missing ship_mode via Series-aligned transform
df.loc[df["ship_mode"].isna() & (df["channel"] == "store"), "ship_mode"] = "in_store"
df["ship_mode"] = df["ship_mode"].fillna(
    df.groupby(["channel", "return_window_days"], sort=False)["ship_mode"].transform(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "standard"
    )
).fillna("standard")

# 3. Relationally impute remaining missing payment_method from customer history & channel/region mode
cust_pay_mode = df.groupby("customer_id", sort=False)["payment_method"].transform("first")
df["payment_method"] = df["payment_method"].fillna(cust_pay_mode)
df["payment_method"] = df["payment_method"].fillna(
    df.groupby(["channel", "cust_region"], sort=False)["payment_method"].transform(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "card"
    )
).fillna("card")

# 4. Relationally impute tz_hint by store_id and customer region
store_tz_mode = df.groupby("store_id", sort=False)["tz_hint"].transform("first")
df["tz_hint"] = df["tz_hint"].fillna(store_tz_mode).fillna("UTC")

# 5. Relationally impute session_clicks by (device_type, channel) median
dev_chan_clicks = df.groupby(["device_type", "channel"], sort=False)["session_clicks"].transform("median")
df["session_clicks"] = df["session_clicks"].fillna(dev_chan_clicks).fillna(9.0)

# 6. Coupon code handling
df["has_coupon"] = df["coupon_code"].notna().astype(np.int8)
df["coupon_code"] = df["coupon_code"].fillna("NO_COUPON")

# 7. Analytical Imputation of freight_cost using physical weight, ship_mode, channel & fuel_surcharge
df["freight_cost_missing"] = df["freight_cost"].isna().astype(np.int8)
df["line_total_weight_kg"] = df["weight_kg"] * df["quantity"]
df.loc[df["freight_cost"].isna() & (df["ship_mode"] == "in_store"), "freight_cost"] = 0.0

df["freight_per_effective_kg"] = df["freight_cost"] / (
    (df["line_total_weight_kg"] + 1.0) * (1.0 + df["fuel_surcharge"])
)
group_freight_rate = df.groupby(["ship_mode", "channel", "warehouse_zone"], sort=False)[
    "freight_per_effective_kg"
].transform("median")
imputed_freight = (
    group_freight_rate * (df["line_total_weight_kg"] + 1.0) * (1.0 + df["fuel_surcharge"])
).clip(lower=0.0)
df["freight_cost"] = df["freight_cost"].fillna(imputed_freight).fillna(df["freight_cost"].median())
df.drop(columns=["freight_per_effective_kg"], inplace=True)


# ==============================================================================
# CELL 5: DEEP DOMAIN FEATURE EXTRACTION & INTERACTION ENGINEERING
# ==============================================================================
print("\nGenerating multi-table domain, behavioral, and temporal features...")

# --- 5A. Pricing, Elasticity & Financial Ratios ---
df["exact_line_revenue"] = df["unit_price"] * df["quantity"]
df["list_line_revenue"] = df["list_price"] * df["quantity"]
df["dollar_savings_line"] = (df["list_line_revenue"] - df["exact_line_revenue"]).clip(lower=0.0)
df["unit_to_list_ratio"] = df["unit_price"] / (df["list_price"] + 1e-4)
df["order_val_rounding_diff"] = df["order_value"] - df["exact_line_revenue"]
df["freight_to_order_val_ratio"] = df["freight_cost"] / (df["order_value"] + 1.0)
df["freight_to_unit_price_ratio"] = df["freight_cost"] / (df["unit_price"] + 0.1)
df["total_landed_cost"] = df["order_value"] + df["freight_cost"]
df["effective_unit_landed_cost"] = df["total_landed_cost"] / df["quantity"]
df["is_high_discount"] = (df["discount"] >= 0.40).astype(np.int8)
df["is_free_shipping"] = (df["freight_cost"] == 0).astype(np.int8)

# --- 5B. Order Basket Composition & Intra-Order Aggregations (via order_id) ---
df["line_pos_ratio"] = df["line_no"] / (df["items_in_order"] + 1e-4)
df["is_first_line"] = (df["line_no"] == 1).astype(np.int8)
df["is_last_line"] = (df["line_no"] == df["items_in_order"]).astype(np.int8)
df["is_single_item_order"] = (df["items_in_order"] == 1).astype(np.int8)

order_grp = df.groupby("order_id", sort=False)
df["order_observed_lines"] = order_grp["order_line_id"].transform("count")
df["order_coverage_ratio"] = df["order_observed_lines"] / (df["items_in_order"] + 1e-4)
df["order_total_spend"] = order_grp["order_value"].transform("sum")
df["order_total_qty"] = order_grp["quantity"].transform("sum")
df["order_mean_unit_price"] = order_grp["unit_price"].transform("mean")
df["order_max_unit_price"] = order_grp["unit_price"].transform("max")
df["order_nunique_cats"] = order_grp["category"].transform("nunique")
df["order_nunique_products"] = order_grp["product_id"].transform("nunique")

# Share of the current item within its basket (Bracketing / Wardrobing indicators)
df["line_spend_share_of_order"] = df["order_value"] / (df["order_total_spend"] + 1e-4)
df["line_qty_share_of_order"] = df["quantity"] / (df["order_total_qty"] + 1e-4)
df["line_price_vs_order_mean"] = df["unit_price"] / (df["order_mean_unit_price"] + 1e-4)
df["line_price_vs_order_max"] = df["unit_price"] / (df["order_max_unit_price"] + 1e-4)

# Detect "Bracketing" behavior: buying multiple items of the same category in the same order
df["order_same_cat_lines"] = df.groupby(["order_id", "category"], sort=False)["order_line_id"].transform("count")
df["is_category_bracketed"] = (df["order_same_cat_lines"] > 1).astype(np.int8)
df["order_same_prod_lines"] = df.groupby(["order_id", "product_id"], sort=False)["order_line_id"].transform("count")

# --- 5C. Customer Behavioral & RFM Profiles ---
cust_grp = df.groupby("customer_id", sort=False)
df["cust_total_lines"] = cust_grp["order_line_id"].transform("count")
df["cust_nunique_orders"] = cust_grp["order_id"].transform("nunique")
df["cust_mean_order_val"] = cust_grp["order_value"].transform("mean")
df["cust_std_order_val"] = cust_grp["order_value"].transform("std").fillna(0.0)
df["cust_mean_discount"] = cust_grp["discount"].transform("mean")
df["cust_mean_qty"] = cust_grp["quantity"].transform("mean")
df["cust_nunique_categories"] = cust_grp["category"].transform("nunique")
df["cust_nunique_stores"] = cust_grp["store_id"].transform("nunique")
df["cust_gift_wrap_rate"] = cust_grp["gift_wrap"].transform("mean")
df["cust_coupon_rate"] = cust_grp["has_coupon"].transform("mean")

df["order_val_vs_cust_mean"] = df["order_value"] / (df["cust_mean_order_val"] + 1.0)
df["qty_vs_cust_mean"] = df["quantity"] / (df["cust_mean_qty"] + 0.1)
df["discount_vs_cust_mean"] = df["discount"] - df["cust_mean_discount"]

# --- 5D. Product Popularity, Price Dispersion & Supplier Dynamics ---
prod_grp = df.groupby("product_id", sort=False)
df["prod_tx_count"] = prod_grp["order_line_id"].transform("count")
df["prod_mean_sold_price"] = prod_grp["unit_price"].transform("mean")
df["prod_mean_qty_ordered"] = prod_grp["quantity"].transform("mean")
df["prod_mean_discount"] = prod_grp["discount"].transform("mean")
df["unit_price_vs_prod_mean"] = df["unit_price"] / (df["prod_mean_sold_price"] + 1e-4)

supp_grp = df.groupby("supplier_code", sort=False)
df["supp_tx_count"] = supp_grp["order_line_id"].transform("count")
df["supp_nunique_products"] = supp_grp["product_id"].transform("nunique")

# --- 5E. Store & Cross-Table Alignment Features ---
store_grp = df.groupby("store_id", sort=False)
df["store_tx_count"] = store_grp["order_line_id"].transform("count")
df["store_mean_order_val"] = store_grp["order_value"].transform("mean")
df["is_same_region"] = (df["cust_region"] == df["store_region"]).astype(np.int8)
df["is_same_channel"] = (df["channel"] == df["store_channel"]).astype(np.int8)

# --- 5F. Temporal & Cyclical Features (Generalization-Safe for Months 25-30) ---
df["order_hour"] = df["order_ts"].dt.hour.fillna(12).astype(np.int8)
df["order_dayofweek"] = df["order_ts"].dt.dayofweek.fillna(0).astype(np.int8)
df["order_day"] = df["order_ts"].dt.day.fillna(15).astype(np.int8)
df["order_month"] = df["order_ts"].dt.month.fillna(6).astype(np.int8)
df["is_weekend"] = (df["order_dayofweek"] >= 5).astype(np.int8)
df["is_late_night_order"] = df["order_hour"].isin([23, 0, 1, 2, 3, 4, 5]).astype(np.int8)

# Cyclical trigonometric encoding
df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12.0)
df["dow_sin"] = np.sin(2 * np.pi * df["order_dayofweek"] / 7.0)
df["dow_cos"] = np.cos(2 * np.pi * df["order_dayofweek"] / 7.0)
df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24.0)

# Relative Tenure Deltas
df["cust_tenure_days"] = (df["order_ts"] - df["signup_date"]).dt.days.fillna(365).clip(lower=-30, upper=5000)
df["prod_age_days"] = (df["order_ts"] - df["launch_date"]).dt.days.fillna(365).clip(lower=0, upper=5000)
df["store_age_days"] = (df["order_ts"] - df["opened_date"]).dt.days.fillna(1000).clip(lower=0, upper=10000)
df["reviews_per_prod_year"] = df["review_count"] / ((df["prod_age_days"] / 365.25) + 0.1)
df["clicks_per_item_in_order"] = df["session_clicks"] / (df["items_in_order"] + 1.0)
df["return_window_to_freight"] = df["return_window_days"] / (df["freight_cost"] + 1.0)

# Save order_ts for temporal validation & recency weights before dropping raw datetime objects
order_ts_train = df["order_ts"].iloc[:n_train].copy()
order_ids_train = df["order_id"].iloc[:n_train].to_numpy(dtype=object)
submission_ids = df["order_line_id"].iloc[n_train:].to_numpy(dtype=object)

DROP_COLS = [
    "order_line_id", "order_id", "order_ts", "week_start",
    "signup_date", "record_updated_at", "launch_date", "opened_date",
    "cpi", "unemployment",
]
df.drop(columns=DROP_COLS, inplace=True)

# Force categorical columns to standard Python object dtype for Pandas 3.0 + CatBoost compatibility
CAT_COLS = [
    "customer_id", "product_id", "store_id", "supplier_code", "coupon_code",
    "tz_hint", "channel", "payment_method", "ship_mode", "promo_channel_code",
    "warehouse_zone", "picker_shift", "device_type", "cust_region",
    "customer_type", "age_band", "preferred_contact", "loyalty_tier",
    "category", "packaging_type", "color", "store_channel", "store_region", "format",
]
for c in CAT_COLS:
    df[c] = pd.Series(
        df[c].fillna("MISSING").astype(str).to_numpy(dtype=object),
        index=df.index,
        dtype=object,
    )

# --- 5G. Smart Outlier Management (Quantile Winsorization on Heavy-Tailed Numerics) ---
SKEWED_CONTINUOUS_COLS = [
    "unit_price", "quantity", "freight_cost", "order_value",
    "exact_line_revenue", "list_line_revenue", "dollar_savings_line",
    "line_total_weight_kg", "order_total_spend", "order_total_qty",
    "total_landed_cost", "effective_unit_landed_cost", "prod_volume_cm3",
]
for col in SKEWED_CONTINUOUS_COLS:
    q_low = float(df[col].iloc[:n_train].quantile(0.001))
    q_high = float(df[col].iloc[:n_train].quantile(0.999))
    df[f"{col}_winsor"] = df[col].clip(lower=q_low, upper=q_high)

NUM_COLS = [c for c in df.columns if c not in CAT_COLS]
df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], np.nan).fillna(0.0)
df = reduce_mem_usage(df, "Engineered Full Dataset")

X_train_full = df.iloc[:n_train].copy().reset_index(drop=True)
X_test_full = df.iloc[n_train:].copy().reset_index(drop=True)
del df
gc.collect()


# ==============================================================================
# CELL 6: OUT-OF-FOLD TARGET ENCODING + PREPROCESSING TRANSFORMERS +
#         POLYNOMIAL GRADIENT DESCENT LOGISTIC REGRESSION (L1/L2 ELASTICNET)
# ==============================================================================
print("\nBuilding Out-Of-Fold Smoothed Target Encodings & Polynomial ElasticNet SGD Model...")

N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS)
folds = list(sgkf.split(X_train_full, y_train, groups=order_ids_train))

# --- 6A. Out-Of-Fold Smoothed Target Encoding (Grouped by order_id to prevent leakage) ---
TE_COLS = [
    "product_id", "customer_id", "store_id", "supplier_code",
    "category", "cust_region", "store_region", "channel",
    "ship_mode", "payment_method", "return_window_days", "coupon_code",
]
global_prior = float(y_train.mean())
SMOOTHING = 20.0

for col in TE_COLS:
    oof_te = np.full(n_train, global_prior, dtype=np.float32)
    test_te_accum = np.zeros(len(X_test_full), dtype=np.float32)

    for tr_idx, val_idx in folds:
        tr_series = X_train_full[col].iloc[tr_idx]
        tr_target = pd.Series(y_train[tr_idx], index=tr_series.index)
        stats = tr_target.groupby(tr_series, sort=False).agg(["count", "mean"])
        smoothed = (stats["count"] * stats["mean"] + SMOOTHING * global_prior) / (
            stats["count"] + SMOOTHING
        )
        oof_te[val_idx] = (
            X_train_full[col].iloc[val_idx].map(smoothed).fillna(global_prior).to_numpy(dtype=np.float32)
        )
        test_te_accum += (
            X_test_full[col].map(smoothed).fillna(global_prior).to_numpy(dtype=np.float32) / N_SPLITS
        )

    X_train_full[f"te_{col}"] = oof_te
    X_test_full[f"te_{col}"] = test_te_accum

# --- 6B. FunctionTransformer, PowerTransformer, StandardScaler, MinMaxScaler & Polynomial SGD ---
POLY_BASE_COLS = [
    "unit_price_winsor", "quantity_winsor", "discount", "freight_cost_winsor",
    "order_value_winsor", "return_window_days", "items_in_order", "session_clicks",
    "weight_kg", "seller_rating", "te_category", "te_product_id",
    "te_store_id", "te_ship_mode", "te_return_window_days",
]

log_transformer = FunctionTransformer(
    lambda x: np.log1p(np.clip(x, a_min=0.0, a_max=None)), validate=False
)
pt_scaler = PowerTransformer(method="yeo-johnson", standardize=False)
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler(feature_range=(-1.0, 1.0))
poly_gen = PolynomialFeatures(degree=2, include_bias=False)

X_tr_num_raw = X_train_full[POLY_BASE_COLS].fillna(0.0).to_numpy(dtype=np.float32)
X_te_num_raw = X_test_full[POLY_BASE_COLS].fillna(0.0).to_numpy(dtype=np.float32)

sample_idx = np.random.choice(n_train, size=min(200_000, n_train), replace=False)
X_tr_log = log_transformer.fit_transform(X_tr_num_raw)
X_te_log = log_transformer.transform(X_te_num_raw)

pt_scaler.fit(X_tr_log[sample_idx])
X_tr_pt = pt_scaler.transform(X_tr_log)
X_te_pt = pt_scaler.transform(X_te_log)

X_tr_std = std_scaler.fit_transform(X_tr_pt)
X_te_std = std_scaler.transform(X_te_pt)

X_tr_norm = minmax_scaler.fit_transform(X_tr_std)
X_te_norm = minmax_scaler.transform(X_te_std)

X_tr_poly = poly_gen.fit_transform(X_tr_norm).astype(np.float32)
X_te_poly = poly_gen.transform(X_te_norm).astype(np.float32)

oof_sgd_preds = np.zeros(n_train, dtype=np.float32)
test_sgd_preds = np.zeros(len(X_test_full), dtype=np.float32)

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    sgd_logreg = SGDClassifier(
        loss="log_loss",
        penalty="elasticnet",
        alpha=1e-4,
        l1_ratio=0.25,
        max_iter=30,
        tol=1e-4,
        learning_rate="optimal",
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=SEED + fold_idx,
    )
    sgd_logreg.fit(X_tr_poly[tr_idx], y_train[tr_idx])
    oof_sgd_preds[val_idx] = sgd_logreg.predict_proba(X_tr_poly[val_idx])[:, 1]
    test_sgd_preds += sgd_logreg.predict_proba(X_te_poly)[:, 1] / N_SPLITS

sgd_oof_auc = roc_auc_score(y_train, oof_sgd_preds)
print(f"[Track B] Polynomial + ElasticNet SGD Logistic Regression OOF ROC-AUC: {sgd_oof_auc:.5f}")

X_train_full["poly_elasticnet_sgd_prob"] = oof_sgd_preds
X_test_full["poly_elasticnet_sgd_prob"] = test_sgd_preds

del X_tr_num_raw, X_te_num_raw, X_tr_log, X_te_log, X_tr_pt, X_te_pt, X_tr_std, X_te_std, X_tr_norm, X_te_norm, X_tr_poly, X_te_poly
gc.collect()


# ==============================================================================
# CELL 7: TEMPORAL VALIDATION SPLIT & OPTUNA HYPERPARAMETER TUNING FOR CATBOOST
# ==============================================================================
print("\nConfiguring Temporal Holdout Split & Temporal Recency Weights...")

min_ts = order_ts_train.min()
max_ts = order_ts_train.max()
months_elapsed = ((order_ts_train - min_ts).dt.days / 30.4375).fillna(12.0).to_numpy(dtype=np.float32)

# Temporal validation split: last 4 months of training data (Months 21-24) mimics Months 25-30 test horizon
temporal_cutoff = max_ts - pd.Timedelta(days=120)
temp_tr_mask = (order_ts_train < temporal_cutoff).to_numpy()
temp_val_mask = (order_ts_train >= temporal_cutoff).to_numpy()

print(f"Temporal Train Rows (Months 1-20) : {temp_tr_mask.sum():,}")
print(f"Temporal Valid Rows (Months 21-24): {temp_val_mask.sum():,}")

ALL_FEATURES = list(X_train_full.columns)
CAT_INDICES = [ALL_FEATURES.index(c) for c in CAT_COLS]

# Ensure all CAT_COLS are strictly object dtype right before Pool creation
for c in CAT_COLS:
    X_train_full[c] = pd.Series(X_train_full[c].to_numpy(dtype=object), index=X_train_full.index, dtype=object)
    X_test_full[c] = pd.Series(X_test_full[c].to_numpy(dtype=object), index=X_test_full.index, dtype=object)

tr_indices = np.where(temp_tr_mask)[0]
val_indices = np.where(temp_val_mask)[0]
if len(tr_indices) > 450_000:
    tr_sub_idx = np.random.choice(tr_indices, size=450_000, replace=False)
else:
    tr_sub_idx = tr_indices

dtrain_optuna = Pool(
    data=X_train_full.iloc[tr_sub_idx],
    label=y_train[tr_sub_idx],
    cat_features=CAT_INDICES,
    weight=np.exp(0.025 * months_elapsed[tr_sub_idx]),
)
dvalid_optuna = Pool(
    data=X_train_full.iloc[val_indices],
    label=y_train[val_indices],
    cat_features=CAT_INDICES,
)


def optuna_catboost_objective(trial: optuna.Trial) -> float:
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.10, log=True),
        "depth": trial.suggest_int("depth", 5, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.5, 5.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
        "border_count": trial.suggest_categorical("border_count", [128, 254]),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [10, 25]),
        "task_type": "GPU" if USE_GPU else "CPU",
        "random_seed": SEED,
        "verbose": False,
    }
    model = CatBoostClassifier(**params)
    model.fit(
        dtrain_optuna,
        eval_set=dvalid_optuna,
        early_stopping_rounds=100,
        verbose=False,
    )
    val_preds = model.predict_proba(dvalid_optuna)[:, 1]
    return roc_auc_score(y_train[val_indices], val_preds)


N_OPTUNA_TRIALS = 12
print(f"\nRunning {N_OPTUNA_TRIALS} Optuna trials on Temporal Validation Split...")
study = optuna.create_study(direction="maximize")
study.optimize(optuna_catboost_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best_cb_params = study.best_params
print(f"\nBest Temporal Validation ROC-AUC: {study.best_value:.5f}")
print("Best Optuna CatBoost Hyperparameters:", best_cb_params)

del dtrain_optuna, dvalid_optuna
gc.collect()


# ==============================================================================
# CELL 8: FINAL TRAINING WITH OPTUNA-TUNED GPU CATBOOST & SUBMISSION GENERATION
# ==============================================================================
final_cb_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 3500,
    "learning_rate": best_cb_params["learning_rate"],
    "depth": best_cb_params["depth"],
    "l2_leaf_reg": best_cb_params["l2_leaf_reg"],
    "random_strength": best_cb_params["random_strength"],
    "bagging_temperature": best_cb_params["bagging_temperature"],
    "border_count": best_cb_params["border_count"],
    "one_hot_max_size": best_cb_params["one_hot_max_size"],
    "task_type": "GPU" if USE_GPU else "CPU",
    "random_seed": SEED,
    "verbose": 250,
}

sample_weights_full = np.exp(0.025 * months_elapsed).astype(np.float32)

oof_cb_preds = np.zeros(n_train, dtype=np.float32)
test_pool = Pool(data=X_test_full, cat_features=CAT_INDICES)
test_cb_preds = np.zeros(len(X_test_full), dtype=np.float32)

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n--- Training Optuna-Tuned GPU CatBoost | Fold {fold_idx + 1}/{N_SPLITS} ---")
    tr_pool = Pool(
        data=X_train_full.iloc[tr_idx],
        label=y_train[tr_idx],
        cat_features=CAT_INDICES,
        weight=sample_weights_full[tr_idx],
    )
    val_pool = Pool(
        data=X_train_full.iloc[val_idx],
        label=y_train[val_idx],
        cat_features=CAT_INDICES,
    )

    cb_model = CatBoostClassifier(**final_cb_params)
    cb_model.fit(
        tr_pool,
        eval_set=val_pool,
        early_stopping_rounds=150,
        use_best_model=True,
    )

    fold_val_preds = cb_model.predict_proba(val_pool)[:, 1]
    oof_cb_preds[val_idx] = fold_val_preds
    fold_auc = roc_auc_score(y_train[val_idx], fold_val_preds)
    print(f"Fold {fold_idx + 1} Best Iteration: {cb_model.get_best_iteration()} | Validation ROC-AUC: {fold_auc:.5f}")

    test_cb_preds += cb_model.predict_proba(test_pool)[:, 1] / N_SPLITS

    if fold_idx == N_SPLITS - 1:
        fi_df = pd.DataFrame({
            "feature": ALL_FEATURES,
            "importance": cb_model.get_feature_importance(),
        }).sort_values("importance", ascending=False)
        print("\nTop 20 Most Important Features:")
        print(fi_df.head(20).to_string(index=False))

    del tr_pool, val_pool, cb_model
    gc.collect()

overall_oof_auc = roc_auc_score(y_train, oof_cb_preds)
temporal_oof_auc = roc_auc_score(y_train[temp_val_mask], oof_cb_preds[temp_val_mask])
print("\n" + "=" * 70)
print(f"Overall StratifiedGroupKFold OOF ROC-AUC       : {overall_oof_auc:.5f}")
print(f"Temporal Holdout (Months 21-24) OOF ROC-AUC    : {temporal_oof_auc:.5f}  <-- Matches Kaggle LB")
print("=" * 70)

sub = pd.DataFrame({
    "order_line_id": submission_ids,
    "returned": np.clip(test_cb_preds, 1e-6, 1.0 - 1e-6),
})
sub.to_csv("submission12.csv", index=False)
print(f"\nSaved primary submission.csv | Shape: {sub.shape} | Nulls: {sub['returned'].isna().sum()}")
print(sub.head(10))

Data Directory : C:\Users\Vishesh\OneDrive\coding\Python\optiml
GPU Available  : True

Loading supplementary and transaction tables...
Loaded train.csv: (1400005, 27)
Loaded test.csv : (479995, 26)
Target Return Rate in Train: 0.18053
Supplementary tables cleaned and enriched successfully.

Combined transaction table shape: (1880000, 26)
Corrected 37,806 rows with x100 unit_price cents anomaly.

Generating multi-table domain, behavioral, and temporal features...
[Engineered Full Dataset] Memory reduced: 2196.3 MB -> 1181.5 MB

Building Out-Of-Fold Smoothed Target Encodings & Polynomial ElasticNet SGD Model...
[Track B] Polynomial + ElasticNet SGD Logistic Regression OOF ROC-AUC: 0.49998

Configuring Temporal Holdout Split & Temporal Recency Weights...
Temporal Train Rows (Months 1-20) : 1,179,486
Temporal Valid Rows (Months 21-24): 220,519

Running 12 Optuna trials on Temporal Validation Split...


Best trial: 11. Best value: 0.705854: 100%|██████████| 12/12 [12:24<00:00, 62.07s/it]



Best Temporal Validation ROC-AUC: 0.70585
Best Optuna CatBoost Hyperparameters: {'learning_rate': 0.09798513573698472, 'depth': 5, 'l2_leaf_reg': 2.3636182619225075, 'random_strength': 2.9760164039913364, 'bagging_temperature': 0.17638084713561186, 'border_count': 128, 'one_hot_max_size': 25}

--- Training Optuna-Tuned GPU CatBoost | Fold 1/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.4990458	best: 0.4990458 (0)	total: 79.3ms	remaining: 4m 37s
250:	test: 0.7289414	best: 0.7289414 (250)	total: 15.2s	remaining: 3m 17s
500:	test: 0.7375382	best: 0.7375486 (498)	total: 30.5s	remaining: 3m 2s
750:	test: 0.7386672	best: 0.7386672 (750)	total: 46.4s	remaining: 2m 49s
1000:	test: 0.7398809	best: 0.7398813 (999)	total: 1m 2s	remaining: 2m 36s
1250:	test: 0.7403000	best: 0.7403426 (1229)	total: 1m 18s	remaining: 2m 20s
1500:	test: 0.7414503	best: 0.7414770 (1488)	total: 1m 34s	remaining: 2m 5s
bestTest = 0.7418069839
bestIteration = 1584
Shrink model to first 1585 iterations.
Fold 1 Best Iteration: 1584 | Validation ROC-AUC: 0.74181

--- Training Optuna-Tuned GPU CatBoost | Fold 2/5 ---


...

0:	test: 0.4994673	best: 0.4994673 (0)	total: 65.1ms	remaining: 3m 47s
250:	test: 0.7172852	best: 0.7173039 (247)	total: 15.1s	remaining: 3m 15s
500:	test: 0.7415140	best: 0.7417033 (442)	total: 30.8s	remaining: 3m 4s
750:	test: 0.7439241	best: 0.7439287 (748)	total: 46.5s	remaining: 2m 50s
1000:	test: 0.7444275	best: 0.7444327 (998)	total: 1m 2s	remaining: 2m 36s
1250:	test: 0.7450452	best: 0.7450572 (1246)	total: 1m 18s	remaining: 2m 20s
bestTest = 0.7451644838
bestIteration = 1260
Shrink model to first 1261 iterations.
Fold 3 Best Iteration: 1260 | Validation ROC-AUC: 0.74516

--- Training Optuna-Tuned GPU CatBoost | Fold 4/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.4989443	best: 0.4989443 (0)	total: 78.2ms	remaining: 4m 33s
250:	test: 0.7138550	best: 0.7138551 (249)	total: 15.5s	remaining: 3m 20s
500:	test: 0.7401642	best: 0.7401642 (500)	total: 31.3s	remaining: 3m 7s
bestTest = 0.7401642203
bestIteration = 500
Shrink model to first 501 iterations.
Fold 4 Best Iteration: 500 | Validation ROC-AUC: 0.74016

--- Training Optuna-Tuned GPU CatBoost | Fold 5/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.4989013	best: 0.4989013 (0)	total: 67ms	remaining: 3m 54s
250:	test: 0.7320448	best: 0.7320448 (250)	total: 15.5s	remaining: 3m 20s
500:	test: 0.7420478	best: 0.7434095 (428)	total: 31.2s	remaining: 3m 6s
bestTest = 0.7434094548
bestIteration = 428
Shrink model to first 429 iterations.
Fold 5 Best Iteration: 428 | Validation ROC-AUC: 0.74341

Top 20 Most Important Features:
                  feature  importance
            te_product_id   29.338929
               product_id   27.590616
           te_customer_id   16.583788
              customer_id   13.725373
            prod_tx_count    3.307063
         cust_total_lines    3.031995
      cust_nunique_orders    1.759429
line_spend_share_of_order    1.023176
   order_nunique_products    0.512506
              te_store_id    0.356098
           items_in_order    0.301463
         te_supplier_code    0.177544
           te_coupon_code    0.143812
  line_qty_share_of_order    0.124699
                    color    0.114219
    